# 03 — Inference-Time Diagnostics: SBI, IPI and Computational Tax

Derives the three diagnostics that let a practitioner decide, before trusting a
COMET score, whether the encoder is carrying the target script well.

| Diagnostic | Formula | Ideal | Role |
|---|---|---|---|
| **SBI** | E_i[TP_i / IP_i] | 1.0 | Encoder pieces spent per unit of information recovered. Flag ≥ 3.0 |
| **IPI** | \|IP − 1.0\| | 0.0 | Distance from English-equivalent parity; drives zone classification |
| **Computational Tax** | LP × EP | 1.0 | Joint multiplicative overhead of romanisation |

with **LP** = TP_rom / TP_nat (Length Penalty) and **EP** = IP_nat / IP_rom
(Entropy Penalty). Decomposing in log space, `ln Tax = ln LP + ln EP`, gives each
component's share independently of the scale of Tax.

Zones: **Parity** (IPI < 0.05), **Burden** (0.05–0.70), **Paradox** (> 0.70).

**Inputs:** `../data/indic/indic_parity_xlmr.xlsx`, `../data/latin/wmt24_ende_enes_metrics.xlsx`
**Outputs:**
- `../results/tables/diagnostics_sbi_ipi_tax.csv`
- `../results/tables/latin_controls.csv`

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)


# ── Diagnostic thresholds (empirically calibrated) ───────────────────────────
SBI_FLAG    = 3.0    # SBI >= 3.0  -> unreliability flag
IPI_PARITY  = 0.05   # IPI < 0.05  -> Parity zone
IPI_PARADOX = 0.70   # IPI > 0.70  -> Paradox zone; in between -> Burden

## Step 1 — Load the Workbook

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats

## Step 2 — SBI, IPI, LP, EP and Tax

SBI is a *per-sentence* ratio averaged over sentences, not a ratio of means —
the two differ, and the paper uses the former. IPI, LP and EP are computed from
language-level mean TP and IP.

In [ ]:
def zone(ipi):
    if ipi < IPI_PARITY:
        return "Parity"
    if ipi > IPI_PARADOX:
        return "Paradox"
    return "Burden"


rows = []
print("Diagnostics per language (N = 1,400 each)")
print(f"{'Lang':>5}  {'SBI':>6}  {'P(SBI>=3)':>10}  {'IPI_nat':>8}  {'IPI_rom':>8}  "
      f"{'zone_nat':>9}  {'zone_rom':>9}  {'LP':>5}  {'EP':>5}  {'Tax':>5}  {'EP%':>6}")
print("-" * 96)
for lang in LANG_ORDER:
    d = full[lang]
    sbi_sent = d[COL_TP_NAT] / d[COL_IP_NAT].replace(0, np.nan)
    ip_n, ip_r = d[COL_IP_NAT].mean(), d[COL_IP_ROM].mean()
    tp_n, tp_r = d[COL_TP_NAT].mean(), d[COL_TP_ROM].mean()
    lp, ep = tp_r / tp_n, ip_n / ip_r
    tax = lp * ep
    ep_pct = 100 * np.log(ep) / np.log(tax)
    ipi_n, ipi_r = abs(ip_n - 1), abs(ip_r - 1)
    rows.append(dict(lang=lang, sbi_nat=sbi_sent.mean(),
                     p_sbi_ge3_pct=100 * (sbi_sent >= SBI_FLAG).mean(),
                     ipi_nat=ipi_n, ipi_rom=ipi_r,
                     zone_nat=zone(ipi_n), zone_rom=zone(ipi_r),
                     LP=lp, EP=ep, tax=tax, ep_pct=ep_pct))
    print(f"{lang:>5}  {sbi_sent.mean():>6.2f}  "
          f"{100 * (sbi_sent >= SBI_FLAG).mean():>9.1f}%  {ipi_n:>8.3f}  {ipi_r:>8.3f}  "
          f"{zone(ipi_n):>9}  {zone(ipi_r):>9}  {lp:>5.2f}  {ep:>5.2f}  "
          f"{tax:>5.2f}  {ep_pct:>5.1f}%")

diag = pd.DataFrame(rows).set_index("lang")

## Step 3 — Cross-Verify Against the Paper

In [ ]:
# ── SBI ranks the languages exactly as native COMET does ─────────────────────
comet_nat = [full[l][COL_COMET_NAT].mean() for l in LANG_ORDER]
rho_sbi = stats.spearmanr(diag["sbi_nat"].tolist(), comet_nat)[0]

assert abs(diag.loc["GUJ", "sbi_nat"] - 3.45) < 0.01
assert diag.loc["GUJ", "sbi_nat"] >= SBI_FLAG
assert abs(diag.loc["GUJ", "tax"] - 1.75) < 0.01
assert abs(diag.loc["TAM", "tax"] - 5.57) < 0.01
assert (diag["zone_nat"] == "Burden").all(), "all five should be Burden natively"
assert (diag["zone_rom"] == "Paradox").all(), "all five should cross into Paradox"
assert (diag["ep_pct"] > 50).all(), "EP should dominate LP everywhere"

print(f"\u2713 GUJ SBI = {diag.loc['GUJ', 'sbi_nat']:.2f} (paper: 3.45), flag \u2265 3.0 raised")
print(f"\u2713 Tax spans {diag['tax'].min():.2f}\u00d7 (GUJ) to {diag['tax'].max():.2f}\u00d7 (TAM) "
      f"(paper: 1.75\u00d7 - 5.57\u00d7)")
print(f"\u2713 EP share of ln(Tax) spans {diag['ep_pct'].min():.1f}% to "
      f"{diag['ep_pct'].max():.1f}% (paper: 62-75%)")
print("\u2713 All five languages: Burden (native) \u2192 Paradox (romanised)")
print(f"\u2713 Spearman(SBI_nat, COMET_nat) over n=5 languages = {rho_sbi:+.2f} [descriptive]")

## Step 4 — Latin-Script Controls (WMT24)

Two reference points from the WMT24 General MT shared task, computed from the
same `target_xlmr_IP` / `target_xlmr_TP` definitions:

- **ENG-SPA** sits in the Parity zone and anchors the scale.
- **ENG-DEU** uses the Latin alphabet yet lands squarely in the Burden zone,
  inside the Indic native-script range. Fragmentation is therefore driven by
  encoder vocabulary allocation, not by whether the script is Latin.

These two values are what notebook 09 plots as the control markers in Figure 1.

In [ ]:
latin_rows = []
print("Latin-script controls (WMT24)")
print(f"{'Pair':>9}  {'rows':>6}  {'IP':>7}  {'IPI':>7}  {'TP':>7}  {'SBI':>6}  {'zone':>8}")
print("-" * 60)
for sheet, iso in [("German", "DEU"), ("Spanish", "SPA")]:
    d = pd.read_excel(DATA_LATIN, sheet_name=sheet)
    ip = pd.to_numeric(d["target_xlmr_IP"], errors="coerce")
    tp = pd.to_numeric(d["target_xlmr_TP"], errors="coerce")
    ipi = abs(ip.mean() - 1)
    latin_rows.append(dict(pair=f"ENG-{iso}", rows=len(d), ip=ip.mean(), ipi=ipi,
                           tp=tp.mean(), sbi=(tp / ip).mean(), zone=zone(ipi)))
    print(f"{'ENG-' + iso:>9}  {len(d):>6}  {ip.mean():>7.3f}  {ipi:>7.3f}  "
          f"{tp.mean():>7.3f}  {(tp / ip).mean():>6.3f}  {zone(ipi):>8}")

latin = pd.DataFrame(latin_rows).set_index("pair")

assert abs(latin.loc["ENG-DEU", "ipi"] - 0.465) < 0.001
assert abs(latin.loc["ENG-SPA", "ipi"] - 0.009) < 0.001
assert latin.loc["ENG-SPA", "zone"] == "Parity"
assert latin.loc["ENG-DEU", "zone"] == "Burden"
print(f"\n\u2713 ENG-SPA IPI = {latin.loc['ENG-SPA', 'ipi']:.3f} \u2192 Parity (paper: 0.009)")
print(f"\u2713 ENG-DEU IPI = {latin.loc['ENG-DEU', 'ipi']:.3f} \u2192 Burden (paper: 0.465)")
print(f"  ENG-DEU sits inside the Indic native range "
      f"[{diag['ipi_nat'].min():.3f}, {diag['ipi_nat'].max():.3f}] despite Latin script.")

## Step 5 — Save

In [ ]:
p1 = TABLES_DIR / "diagnostics_sbi_ipi_tax.csv"
p2 = TABLES_DIR / "latin_controls.csv"
diag.to_csv(p1)
latin.to_csv(p2)
print(diag.round(3).to_string())
print()
print(latin.round(3).to_string())
print(f"\nSaved \u2192 {p1}")
print(f"Saved \u2192 {p2}")

## Step 6 — Output Manifest

In [ ]:
print("=== Notebook 03 — output manifest ===")
for name in ["diagnostics_sbi_ipi_tax.csv", "latin_controls.csv"]:
    print(f"  {name}")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1